# 构建用于强化学习的 Gym 环境

本 notebook 将基于 hftbacktest 框架和 PMM 策略构建一个自定义的 OpenAI Gym 环境，用于训练强化学习智能体进行市商策略优化。

## 环境设计目标

-   **状态空间**: 订单簿特征、价格变化、持仓状态、风险指标等
-   **动作空间**: 价差调整、订单数量、策略参数调整等
-   **奖励函数**: 基于收益、风险、库存管理等多维度奖励设计
-   **环境**: 基于真实市场数据的高保真模拟环境


In [1]:

import torch
import warnings

# TorchRL相关导入
from tensordict import TensorDict

# 设置警告过滤
warnings.filterwarnings('ignore')

# 设置默认设备（优先级：CUDA > MPS > CPU）
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 使用CUDA GPU加速")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🍎 使用Apple Silicon MPS加速")
else:
    device = torch.device("cpu")
    print("💻 使用CPU")

print(f"环境依赖库导入完成，使用设备: {device}")

🍎 使用Apple Silicon MPS加速
环境依赖库导入完成，使用设备: mps


In [2]:
# 测试 PMM 强化学习环境

# 导入我们构建的PMM环境
from lib.rl_env import create_pmm_env
from hftbacktest import BacktestAsset
import os

# 使用单个测试数据文件
test_data_file = "data/output/solusdt_20250629.npz"

if os.path.exists(test_data_file):
    print(f"✅ 找到测试数据文件: {test_data_file}")
    try:
        print(f"🔄 正在加载数据: {test_data_file}")

        # 按照03_strategy_design.ipynb中的正确方式创建BacktestAsset
        data_asset = (
            BacktestAsset()
            .data([test_data_file])  # 传入数据文件列表
            .linear_asset(1.0)       # 线性资产，合约乘数为1
            .risk_adverse_queue_model()  # 风险规避队列模型
            .no_partial_fill_exchange()  # 不允许部分成交
            .tick_size(0.01)         # 最小价格精度 (0.01 USDT)
            .lot_size(0.001)         # 最小交易数量
            .trading_value_fee_model(-0.00003, 0.0007)  # Binance手续费模型
        )

        print(f"✅ 成功创建BacktestAsset")
        print(f"   使用数据文件: {test_data_file}")
        print(f"   tick_size: {data_asset.tick_size}")
        print(f"   lot_size: {data_asset.lot_size}")

    except Exception as e:
        print(f"❌ 创建BacktestAsset失败: {e}")
        import traceback
        traceback.print_exc()
        data_asset = None
else:
    print(f"❌ 测试数据文件不存在: {test_data_file}")
    print("   请确保数据文件存在，或修改 test_data_file 路径")
    data_asset = None

✅ 找到测试数据文件: data/output/solusdt_20250629.npz
🔄 正在加载数据: data/output/solusdt_20250629.npz
✅ 成功创建BacktestAsset
   使用数据文件: data/output/solusdt_20250629.npz
   tick_size: <built-in method tick_size of BacktestAsset object at 0x12bb28950>
   lot_size: <built-in method lot_size of BacktestAsset object at 0x12bb28950>


## 🎯 PMM强化学习环境说明

### 环境特点
我们成功构建了一个基于PMM策略的TorchRL强化学习环境，具有以下特点：

#### 🎮 **动作空间** (9维连续动作)
1. **bid_spread** (0.0001-0.01): 买入价差，0.01%-1%
2. **ask_spread** (0.0001-0.01): 卖出价差，0.01%-1%
3. **order_refresh_time** (0.5-5.0): 订单刷新时间（秒）
4. **price_deviation_pct** (0.001-0.01): 价格偏差百分比，0.1%-1%
5. **hang_order_time_limit** (5.0-60.0): 挂单时间限制（秒）
6. **max_open_orders** (10-200): 最大挂单数量，整数
7. **take_profit_pct** (0.0005-0.005): 止盈百分比，0.05%-0.5%
8. **stop_loss_pct** (0.001-0.01): 止损百分比，0.1%-1%
9. **order_amount** (1-200): 订单数量（张合约），整数

#### 👁️ **观测空间** (10维状态向量)
1. **标准化中间价格**: 当前市场中间价/100
2. **价差**: (ask-bid)/mid_price  
3. **标准化仓位**: 当前持仓/100
4. **余额比例**: 当前余额/初始余额
5. **价格波动率**: 基于历史价格的标准差
6. **订单簿不平衡**: (ask-bid)/(ask+bid)
7. **活跃订单比例**: 当前订单数/最大订单数
8. **最近盈亏**: 最近损益/初始余额
9. **平均成交时间**: 订单平均成交时间指标
10. **策略效率**: 基于步数和收益的效率指标

#### 🏆 **奖励函数** (多目标优化)
- **收益奖励**: 基于账户余额变化的主要奖励
- **风险惩罚**: 基于仓位大小的二次惩罚，避免过度持仓
- **交易成本**: 活跃订单的线性成本，鼓励高效策略
- **动作稳定性**: 避免过度频繁调整参数的惩罚

### BacktestAsset配置说明
使用链式调用配置BacktestAsset：
```python
data_asset = (
    BacktestAsset()
    .data([data_file])              # 数据文件列表
    .linear_asset(1.0)              # 线性资产，合约乘数
    .risk_adverse_queue_model()     # 队列模型
    .no_partial_fill_exchange()     # 交易所模型
    .tick_size(0.01)               # 价格精度
    .lot_size(0.001)               # 最小交易量
    .trading_value_fee_model(-0.00003, 0.0007)  # 手续费模型
)
```


In [3]:
# 重新运行测试，现在TensorDict问题已修复
if data_asset is not None:
    print("🚀 重新测试PMM强化学习环境...")
    
    try:
        # 创建环境实例
        env = create_pmm_env(
            data_asset=data_asset,
            initial_balance=10000.0,
            max_steps=100,
            device=device.type,
            risk_penalty_weight=0.01,
            transaction_cost_rate=0.0001
        )
        
        print("✅ 环境创建成功")
        
        # 测试环境重置
        print("\n🔄 测试环境重置...")
        reset_td = env.reset()
        print(f"重置后状态形状: {reset_td['observation'].shape}")
        print(f"重置后完成标志: {reset_td['done']}")
        
        # 测试随机动作 - 现在是9维动作空间
        print("\n🎮 测试随机动作...")
        for step in range(3):
            # 生成随机动作（9维）
            random_action = torch.rand(9, device=env.device)
            action_low = torch.tensor([
                0.0001,  # bid_spread
                0.0001,  # ask_spread
                0.5,     # order_refresh_time
                0.001,   # price_deviation_pct
                5.0,     # hang_order_time_limit
                10.0,    # max_open_orders
                0.0005,  # take_profit_pct
                0.001,   # stop_loss_pct
                1.0,     # order_amount
            ], device=env.device)
            action_high = torch.tensor([
                0.01,    # bid_spread
                0.01,    # ask_spread
                5.0,     # order_refresh_time
                0.01,    # price_deviation_pct
                60.0,    # hang_order_time_limit
                200.0,   # max_open_orders
                0.005,   # take_profit_pct
                0.01,    # stop_loss_pct
                200.0,   # order_amount
            ], device=env.device)
            action = action_low + random_action * (action_high - action_low)
            
            # 执行动作
            action_td = TensorDict({"action": action}, batch_size=(), device=env.device)
            step_td = env.step(action_td)
            
            print(f"\n步骤 {step + 1}:")
            print(f"  动作: {action.cpu().numpy()}")
            print(f"  返回的键: {list(step_td.keys())}")
            print(f"  TensorDict内容: {step_td}")
            
            # 尝试不同的键访问方式
            if 'reward' in step_td:
                print(f"  奖励: {step_td['reward'].item():.6f}")
            elif 'next' in step_td and 'reward' in step_td['next']:
                print(f"  奖励: {step_td['next']['reward'].item():.6f}")
            else:
                print("  ⚠️ 找不到奖励键")
                
            if 'done' in step_td:
                print(f"  完成: {step_td['done'].item()}")
            elif 'next' in step_td and 'done' in step_td['next']:
                print(f"  完成: {step_td['next']['done'].item()}")
            else:
                print("  ⚠️ 找不到完成键")
                
            # 检查是否应该结束
            should_break = False
            if 'done' in step_td and step_td['done'].item():
                should_break = True
            elif 'next' in step_td and 'done' in step_td['next'] and step_td['next']['done'].item():
                should_break = True
                
            if should_break:
                print("  ⚠️ 环境提前结束")
                break
        
        print("\n✅ 环境测试完成!")
        env.close()
        print("🔒 环境已关闭")
        
    except Exception as e:
        print(f"❌ 环境测试失败: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("❌ 无法测试环境：数据加载失败")


🚀 重新测试PMM强化学习环境...
✅ 环境创建成功

🔄 测试环境重置...


/Users/zhoudl0605/miniconda3/envs/rl_pmm/lib/python3.13/site-packages/torchrl/data/tensor_specs.py:6705: DeprecationWarning: The BoundedTensorSpec has been deprecated and will be removed in v0.8. Please use Bounded instead.
  warnings.warn(
/Users/zhoudl0605/miniconda3/envs/rl_pmm/lib/python3.13/site-packages/torchrl/data/tensor_specs.py:6705: DeprecationWarning: The CompositeSpec has been deprecated and will be removed in v0.8. Please use Composite instead.
  warnings.warn(
/Users/zhoudl0605/miniconda3/envs/rl_pmm/lib/python3.13/site-packages/torchrl/data/tensor_specs.py:6705: DeprecationWarning: The UnboundedContinuousTensorSpec has been deprecated and will be removed in v0.8. Please use Unbounded instead.
  warnings.warn(


重置后状态形状: torch.Size([10])
重置后完成标志: tensor([False], device='mps:0')

🎮 测试随机动作...

步骤 1:
  动作: [7.3510166e-03 7.5186789e-03 2.9375105e+00 1.8947399e-03 3.4919445e+01
 1.9737598e+02 4.9329828e-03 7.0008752e-03 1.5801868e+02]
  返回的键: ['action', 'next']
  TensorDict内容: TensorDict(
    fields={
        action: Tensor(shape=torch.Size([9]), device=mps:0, dtype=torch.float32, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([10]), device=mps:0, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.float32, is_shared=False),
                terminated: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([]),
            device=mps:0,
            is_shared=False)},
    batch_size=torch.Size([]),
    de

In [4]:
# 创建并测试PMM强化学习环境
import torch
from tensordict import TensorDict

# 确保device变量可用
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

if data_asset is not None:
    print("🚀 开始测试PMM强化学习环境...")

    try:
        # 创建环境实例
        env = create_pmm_env(
            data_asset=data_asset,
            initial_balance=10000.0,
            max_steps=100,  # 测试用较小步数
            device=device.type,
            risk_penalty_weight=0.01,
            transaction_cost_rate=0.0001
        )

        print("✅ 环境创建成功")

        # 检查环境规格
        print("\n📊 环境规格信息:")
        print(f"动作空间: {env.action_spec}")
        print(f"观测空间: {env.observation_spec}")
        print(f"奖励空间: {env.reward_spec}")
        print(f"完成空间: {env.done_spec}")

        # 测试环境重置
        print("\n🔄 测试环境重置...")
        reset_td = env.reset()
        print(f"重置后状态形状: {reset_td['observation'].shape}")
        print(f"重置后观测值: {reset_td['observation']}")
        print(f"重置后完成标志: {reset_td['done']}")

        # 测试几个随机动作
        print("\n🎮 测试随机动作...")
        for step in range(3):
            # 生成随机动作（9维）
            random_action = torch.rand(9, device=env.device)
            # 缩放到动作空间范围
            action_low = torch.tensor([
                0.0001,  # bid_spread
                0.0001,  # ask_spread
                0.5,     # order_refresh_time
                0.001,   # price_deviation_pct
                5.0,     # hang_order_time_limit
                10.0,    # max_open_orders
                0.0005,  # take_profit_pct
                0.001,   # stop_loss_pct
                1.0,     # order_amount
            ], device=env.device)
            action_high = torch.tensor([
                0.01,    # bid_spread
                0.01,    # ask_spread
                5.0,     # order_refresh_time
                0.01,    # price_deviation_pct
                60.0,    # hang_order_time_limit
                200.0,   # max_open_orders
                0.005,   # take_profit_pct
                0.01,    # stop_loss_pct
                200.0,   # order_amount
            ], device=env.device)
            action = action_low + random_action * (action_high - action_low)

            # 执行动作
            action_td = TensorDict(
                {"action": action}, batch_size=(), device=env.device)
            step_td = env.step(action_td)

            print(f"\n步骤 {step + 1}:")
            print(f"  动作: {action.cpu().numpy()}")
            print(f"  返回的键: {list(step_td.keys())}")

            # 调试不同的键访问方式
            if 'reward' in step_td:
                print(f"  奖励: {step_td['reward'].item():.6f}")
            elif 'next' in step_td and 'reward' in step_td['next']:
                print(f"  奖励: {step_td['next']['reward'].item():.6f}")
            else:
                print("  ⚠️ 找不到奖励键")

            if 'observation' in step_td:
                print(f"  观测: {step_td['observation'].cpu().numpy()}")
            elif 'next' in step_td and 'observation' in step_td['next']:
                print(f"  观测: {step_td['next']['observation'].cpu().numpy()}")
            else:
                print("  ⚠️ 找不到观测键")

            if 'done' in step_td:
                print(f"  完成: {step_td['done'].item()}")
            elif 'next' in step_td and 'done' in step_td['next']:
                print(f"  完成: {step_td['next']['done'].item()}")
            else:
                print("  ⚠️ 找不到完成键")

            # 检查是否应该结束
            should_break = False
            if 'done' in step_td and step_td['done'].item():
                should_break = True
            elif 'next' in step_td and 'done' in step_td['next'] and step_td['next']['done'].item():
                should_break = True

            if should_break:
                print("  ⚠️ 环境提前结束")
                break

        print("\n✅ 环境测试完成!")

        # 关闭环境
        env.close()
        print("🔒 环境已关闭")

    except Exception as e:
        print(f"❌ 环境测试失败: {e}")
        import traceback
        traceback.print_exc()

else:
    print("❌ 无法测试环境：数据加载失败")

🚀 开始测试PMM强化学习环境...
✅ 环境创建成功

📊 环境规格信息:
动作空间: BoundedContinuous(
    shape=torch.Size([9]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([9]), device=mps:0, dtype=torch.float32, contiguous=True),
        high=Tensor(shape=torch.Size([9]), device=mps:0, dtype=torch.float32, contiguous=True)),
    device=mps:0,
    dtype=torch.float32,
    domain=continuous)
观测空间: Composite(
    observation: UnboundedContinuous(
        shape=torch.Size([10]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([10]), device=mps:0, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([10]), device=mps:0, dtype=torch.float32, contiguous=True)),
        device=mps:0,
        dtype=torch.float32,
        domain=continuous),
    device=mps:0,
    shape=torch.Size([]))
奖励空间: UnboundedContinuous(
    shape=torch.Size([1]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.float32, contiguous=True),
        hig

/Users/zhoudl0605/miniconda3/envs/rl_pmm/lib/python3.13/site-packages/torchrl/data/tensor_specs.py:6705: DeprecationWarning: The BoundedTensorSpec has been deprecated and will be removed in v0.8. Please use Bounded instead.
  warnings.warn(
/Users/zhoudl0605/miniconda3/envs/rl_pmm/lib/python3.13/site-packages/torchrl/data/tensor_specs.py:6705: DeprecationWarning: The CompositeSpec has been deprecated and will be removed in v0.8. Please use Composite instead.
  warnings.warn(
/Users/zhoudl0605/miniconda3/envs/rl_pmm/lib/python3.13/site-packages/torchrl/data/tensor_specs.py:6705: DeprecationWarning: The UnboundedContinuousTensorSpec has been deprecated and will be removed in v0.8. Please use Unbounded instead.
  warnings.warn(
